# Product photo to a five-second video

Run the cells from top to bottom. This notebook creates **one five-second video** using your Magic Hour account and existing credits. Each rerun of the generation cell creates another project.

[Create an API key](https://magichour.ai/developer) · [Full recipe and recovery instructions](https://docs.magichour.ai/get-started/starter-recipes)

Use media you own or have permission to use. Files selected below are uploaded to this Google Colab runtime and then to Magic Hour. Review generated output before publishing. 


In [ ]:
%pip install -q "magic-hour==0.78.1"


In [ ]:
import os
from getpass import getpass

os.environ["MAGIC_HOUR_API_KEY"] = getpass("Magic Hour API key (hidden): ").strip()
if not os.environ["MAGIC_HOUR_API_KEY"]:
    raise ValueError("Enter an API key before continuing.")


## Upload your input files


In [ ]:
from google.colab import files
from pathlib import Path

print("Upload one product photo (JPG, PNG, or WebP).")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one image before continuing.")
image_filename = next(iter(uploaded))
if Path(image_filename).suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp"}:
    raise ValueError("Use a JPG, PNG, or WebP image.")


## Create one video

This cell uses credits. Save the printed project ID. If polling or downloading fails, use the recovery instructions in the linked guide instead of rerunning this cell. Stopping Colab does not cancel a submitted project.


In [ ]:
import os
from pathlib import Path

from magic_hour import Client

image = Path(image_filename)
if not image.is_file():
    raise SystemExit("Add product.jpg beside this script before running.")

client = Client(token=os.environ["MAGIC_HOUR_API_KEY"])
os.environ.setdefault("MAGIC_HOUR_POLL_INTERVAL", "3")

uploaded_image = client.v1.files.upload_file(file=str(image))
job = client.v1.image_to_video.create(
    assets={"image_file_path": uploaded_image},
    end_seconds=5,
    model="default",
    name="Product photo starter recipe",
    style={
        "prompt": (
            "Slow, subtle camera push-in toward the product. "
            "Keep the product shape, colors, packaging, and background consistent. "
            "Soft studio lighting. No added text or objects."
        )
    },
)
print(f"Project ID: {job.id}", flush=True)

Path("outputs/product-video").mkdir(parents=True, exist_ok=True)
result = client.v1.video_projects.check_result(
    id=job.id,
    wait_for_completion=True,
    download_outputs=True,
    download_directory="outputs/product-video",
)
if result.status != "complete" or not result.downloaded_paths:
    raise SystemExit(f"Project {result.id}: {result.status}; {result.error}")

for path in result.downloaded_paths:
    print(f"Downloaded: {path}")


## Preview and save

Colab storage is temporary. Download the completed video to keep it. Clear all cell outputs before sharing a copy of this notebook; outputs can contain your media and project ID. The API key is entered with a hidden prompt and is not stored in notebook source.


In [ ]:
from IPython.display import Video, display
from google.colab import files

for output_path in result.downloaded_paths:
    display(Video(output_path, embed=True))
    files.download(output_path)
